# FED / Colab Environment Setup for CTSKII JupyterHub

This notebook configures a **Colab-like environment** so FED notebooks (e.g. SFT_LoRA_Adapters) run on CTSKII JupyterHub with the same package versions.

**Prerequisite:** You must have `colab_requirements.txt` in your home folder (`~/colab_requirements.txt`).

> Get it by running [extract_colab_requirements.ipynb](./extract_colab_requirements.ipynb) in Google Colab, then download and upload to JupyterHub.

Run each cell **in order**. After Step 5, restart the server for the kernel to appear.

---

## Detailed Steps Overview

| Step | Action |
|------|--------|
| 0 | Ensure `colab_requirements.txt` is in your home folder |
| 1 | Check if requirements file exists and Python version |
| 2 | Create venv at `~/colab_venv` |
| 3 | Install packages from `~/colab_requirements.txt` |
| 4 | Register kernel **Python (Colab-like)** |
| 5 | Restart server (File → Hub Control Panel → Stop → wait 20s → Start) |
| 6 | Switch kernel in FED notebook to **Python (Colab-like)** |
| 7 | Run FED notebook (e.g. SFT_LoRA_Adapters.ipynb) |

## Step 0 — Put `colab_requirements.txt` in Home

1. Download `colab_requirements.txt` from Colab (or your repo).
2. In JupyterHub: **File → Upload** and upload it to your home folder.
3. Ensure the path is `~/colab_requirements.txt`.

The next cell checks if the file exists.

## Step 1 — Verify Setup and Python Version

In [ ]:
import sys
import shutil
from pathlib import Path

home_req = Path.home() / "colab_requirements.txt"
cwd = Path.cwd()

# Search: home, cwd, cwd/FED/CONVERIONS, cwd/../CONVERIONS
search_paths = [
    home_req,
    cwd / "colab_requirements.txt",
    cwd / "FED" / "CONVERIONS" / "colab_requirements.txt",
    cwd / "CONVERIONS" / "colab_requirements.txt",
    cwd.parent / "colab_requirements.txt",
]

req_file = None
for p in search_paths:
    if p.exists():
        req_file = p
        break

print("Python version:", sys.version)

if req_file is not None:
    if req_file != home_req:
        shutil.copy(req_file, home_req)
        print(f"Found at {req_file} → copied to {home_req}")
    else:
        print(f"Found at {home_req}")
    lines = [l for l in home_req.read_text().splitlines() if l.strip() and not l.startswith("#")]
    print(f"Found {len(lines)} packages in colab_requirements.txt")
else:
    print("ERROR: colab_requirements.txt NOT found.")
    print("Searched:", [str(p) for p in search_paths])
    print("Upload it to your home folder or the notebook directory and re-run.")

## Step 2 — Create Virtual Environment

Creates `~/colab_venv`. This keeps FED packages separate from the default environment.

The venv persists across server restarts.

In [ ]:
import os

venv_path = os.path.expanduser("~/colab_venv")

if not os.path.exists(venv_path):
    os.system(f"python3 -m venv {venv_path}")
    print("Virtual environment created at ~/colab_venv")
else:
    print("Virtual environment already exists at ~/colab_venv")

## Step 3 — Install Packages from colab_requirements.txt

Installs all packages from `~/colab_requirements.txt` into `~/colab_venv`.

This can take several minutes (torch, transformers, etc.).

In [ ]:
import os

req_file = os.path.expanduser("~/colab_requirements.txt")
pip_path = os.path.expanduser("~/colab_venv/bin/pip")

if os.path.exists(req_file):
    os.system(f"{pip_path} install --upgrade pip ipykernel")
    os.system(f"{pip_path} install -r {req_file}")
    print("Packages installed from colab_requirements.txt")
else:
    print("ERROR: colab_requirements.txt not found. Upload it to your home folder.")

## Step 4 — Register as Jupyter Kernel

Registers `~/colab_venv` as kernel **Python (Colab-like)** so you can select it in any notebook.

In [ ]:
import os

python_path = os.path.expanduser("~/colab_venv/bin/python")

os.system(f"{python_path} -m pip install ipykernel")
os.system(f"{python_path} -m ipykernel install --user --name colab_venv --display-name 'Python (Colab-like)'")

print("Kernel 'Python (Colab-like)' registered.")

## Step 5 — Restart Server (Required)

**The new kernel will not appear until you restart the server.**

1. Go to **File → Hub Control Panel**
2. Click **Stop My Server**
3. Wait **~20 seconds**
4. Click **Start Server**

After the server restarts, return to JupyterLab.

## Step 6 — Switch Kernel in Your FED Notebook

1. Open your FED notebook (e.g. `FED/DS1000/SFT_LoRA_Adapters.ipynb`)
2. Go to **Kernel → Change Kernel → Python (Colab-like)**
3. Run your notebook

## Verify GPU (optional)

Run this cell **after** switching to **Python (Colab-like)** kernel to confirm CUDA is available.

In [ ]:
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
    else:
        print("Running on CPU.")
except Exception as e:
    print("torch not available:", e)

## Summary Checklist

- [ ] `colab_requirements.txt` uploaded to home
- [ ] Step 1: Verified file exists
- [ ] Step 2: Created ~/colab_venv
- [ ] Step 3: Installed packages
- [ ] Step 4: Registered kernel
- [ ] Step 5: Restarted server (Stop → wait → Start)
- [ ] Step 6: Switched kernel to **Python (Colab-like)** in FED notebook
- [ ] Run FED notebook